In [1]:
from manim import *
import os
import math

class PositionalEncodingVis(Scene):
    def construct(self):
        # ==========================================
        # 0. SETUP & LAYOUT
        # ==========================================
        self.camera.background_color = "#333333"
        
        # --- Layout Constants ---
        CODE_SCALE = 0.55
        CODE_CENTER = LEFT * 3.6
        VIS_CENTER = RIGHT * 3.5
        
        # VERTICAL ZONES
        HEADER_Y = 3.7            
        SCRATCH_PAD_TOP_Y = 2.5   
        GRID_CENTER_Y = -1.5 
        
        # 1. Generate Python Code
        code_filename = "pos_enc_loop.py"
        pytorch_code = """import torch
import math

d = 4; max_len = 4
pe = torch.zeros(max_len, d)
pos = torch.arange(0, max_len).unsqueeze(1)

# 1. Frequencies (div_term)
div_term = torch.exp(
    torch.arange(0, d, 2) * (-math.log(10000.0) / d)
)

# 2. EVEN (Sine)
pe[:, 0::2] = torch.sin(pos * div_term)

# 3. ODD (Cosine)
pe[:, 1::2] = torch.cos(pos * div_term)
"""
        with open(code_filename, "w") as f:
            f.write(pytorch_code)

        # --- STATIC ELEMENTS ---
        divider = Line(UP*3.8, DOWN*3.8, color=GRAY_C).move_to(RIGHT * 0.0)
        
        # Formula Header
        formula_tex = MathTex(
            r"PE_{(pos, 2i)} = \sin(pos \cdot \omega_i) \\ PE_{(pos, 2i+1)} = \cos(pos \cdot \omega_i)",
            font_size=24, color=YELLOW
        ).move_to(CODE_CENTER + UP * 3.2)

        # Code Block
        code_obj = Code(
            code_filename, 
            tab_width=2, 
            background="window", 
            language="python", 
            formatter_style="monokai",
            paragraph_config={"font": "Monospace"},
            add_line_numbers=True
        ).scale(CODE_SCALE).next_to(formula_tex, DOWN, buff=0.5)
        
        code_title = Text("Implementation", font_size=18, color=GRAY_C)\
            .next_to(code_obj, UP, aligned_edge=LEFT, buff=0.1)

        # Trace Header
        trace_header = Text("Trace Initialization...", font_size=26, color=YELLOW, weight=BOLD)
        trace_header.move_to(VIS_CENTER + UP * HEADER_Y)

        self.add(divider, formula_tex, code_obj, code_title, trace_header)

        # --- Helper: Highlight Code Line ---
        def get_highlight(line_indices, color=YELLOW):
            if isinstance(line_indices, int): line_indices = [line_indices]
            lines = code_obj[2] 
            rects = VGroup()
            for idx in line_indices:
                if idx < len(lines):
                    rect = SurroundingRectangle(lines[idx], color=color, stroke_width=2, buff=0.05).stretch(1.1, 0)
                    rects.add(rect)
            return rects

        curr_hl = VGroup()

        # ==========================================
        # 1. BUILD GRID (Static)
        # ==========================================
        rows, cols = 4, 4
        grid_group = VGroup()
        grid_squares = [[None for _ in range(cols)] for _ in range(rows)]
        grid_origin = VIS_CENTER + UP * GRID_CENTER_Y

        for r in range(rows):
            for c in range(cols):
                sq = Square(side_length=0.6).set_stroke(WHITE, 1)
                grid_squares[r][c] = sq
                grid_group.add(sq)
        
        grid_group.arrange_in_grid(rows=rows, cols=cols, buff=0.05).move_to(grid_origin)

        # Labels
        row_lbls = VGroup()
        for r in range(rows):
            lbl = MathTex(f"p_{{{r}}}", font_size=22, color=GRAY_B).next_to(grid_squares[r][0], LEFT, buff=0.2)
            row_lbls.add(lbl)

        col_lbls = VGroup()
        for c in range(cols):
            lbl = MathTex(f"d_{{{c}}}", font_size=22, color=GRAY_B).next_to(grid_squares[0][c], UP, buff=0.15) 
            col_lbls.add(lbl)

        grid_title = Text("PE Matrix (4x4)", font_size=20, color=GRAY).next_to(grid_group, DOWN, buff=0.15)

        # --- DESCRIPTION LABEL SETUP ---
        # Position: Below title, Aligned to Grid Left, SHIFTED LEFT 4.0
        # Use Tex to allow math mode for p and d
        desc_lbl = Tex(r"Initializing Matrix: Rows = Positions ($p$), Cols = Dimensions ($d$)...", font_size=22, color=WHITE)
        desc_lbl.next_to(grid_title, DOWN, buff=0.2)
        desc_lbl.align_to(grid_group, LEFT).shift(LEFT * 4.0)

        self.play(FadeIn(grid_group), FadeIn(row_lbls), FadeIn(col_lbls), Write(grid_title), Write(desc_lbl))
        self.wait(1.5)

        # ==========================================
        # 2. CALCULATE FREQUENCIES (Detailed)
        # ==========================================
        
        # Update Description
        new_desc = Tex(r"Step 1: Compute frequencies ($\omega$) for each depth dimension pair ($d$).", font_size=22, color=ORANGE)
        new_desc.next_to(grid_title, DOWN, buff=0.2).align_to(grid_group, LEFT).shift(LEFT * 4.0)
        self.play(Transform(desc_lbl, new_desc))

        hl_freq = get_highlight([8, 9, 10]) 
        self.play(ReplacementTransform(curr_hl, hl_freq))
        curr_hl = hl_freq

        new_header = Text("1. Calculate Frequencies (div_term)", font_size=24, color=ORANGE).move_to(VIS_CENTER + UP * HEADER_Y)
        self.play(Transform(trace_header, new_header))

        # Show Formula
        freq_formula = MathTex(r"\omega_i = \frac{1}{10000^{2i/d}}", font_size=32).move_to(VIS_CENTER + UP * SCRATCH_PAD_TOP_Y)
        self.play(Write(freq_formula))
        self.wait(1)

        # --- i=0 ---
        self.play(freq_formula.animate.shift(UP * 0.5).scale(0.8))
        
        calc_i0 = MathTex(r"i=0: \quad \omega_0 = 1.0", font_size=26, color=ORANGE).next_to(freq_formula, DOWN, buff=0.3)
        self.play(Write(calc_i0))
        
        hl_cols_0 = SurroundingRectangle(VGroup(col_lbls[0], col_lbls[1]), color=ORANGE, buff=0.1)
        val_0 = MathTex("1.0", color=ORANGE, font_size=20).next_to(hl_cols_0, UP, buff=0.05)
        self.play(Create(hl_cols_0), FadeIn(val_0))
        self.wait(1)

        # --- i=1 ---
        calc_i1 = MathTex(r"i=1: \quad \omega_1 = 0.01", font_size=26, color=ORANGE).next_to(calc_i0, DOWN, buff=0.2)
        self.play(Write(calc_i1))
        
        hl_cols_1 = SurroundingRectangle(VGroup(col_lbls[2], col_lbls[3]), color=ORANGE, buff=0.1)
        val_1 = MathTex("0.01", color=ORANGE, font_size=20).next_to(hl_cols_1, UP, buff=0.05)
        self.play(Create(hl_cols_1), FadeIn(val_1))
        self.wait(1)

        # Cleanup Frequencies
        self.play(
            FadeOut(freq_formula), FadeOut(calc_i0), FadeOut(calc_i1), 
            FadeOut(hl_cols_0), FadeOut(hl_cols_1)
        )
        
        # Persist values
        perm_val_0 = val_0.copy().scale(0.8).next_to(col_lbls[0], UP, buff=0.05)
        perm_val_1 = val_0.copy().scale(0.8).next_to(col_lbls[1], UP, buff=0.05)
        perm_val_2 = val_1.copy().scale(0.8).next_to(col_lbls[2], UP, buff=0.05)
        perm_val_3 = val_1.copy().scale(0.8).next_to(col_lbls[3], UP, buff=0.05)
        
        self.play(
            Transform(val_0, VGroup(perm_val_0, perm_val_1)),
            Transform(val_1, VGroup(perm_val_2, perm_val_3))
        )
        freq_vals = [1.0, 1.0, 0.01, 0.01]

        # ==========================================
        # 3. LOOP POSITIONS
        # ==========================================
        
        for pos_idx in [1, 2]: 
            # Header
            pos_title = Text(f"Position p = {pos_idx}", font_size=24, color=YELLOW).move_to(VIS_CENTER + UP * HEADER_Y)
            self.play(Transform(trace_header, pos_title))
            
            # Highlight Row
            row_rect = SurroundingRectangle(VGroup(*grid_squares[pos_idx]), color=YELLOW, buff=0.08)
            self.play(Create(row_rect))

            # --- SINE ---
            # Update Description
            desc_sin = Tex(rf"Step 2: Fill Row $p_{{{pos_idx}}}$ at EVEN cols ($d_0, d_2$) using Sine.", font_size=22, color=BLUE)
            desc_sin.next_to(grid_title, DOWN, buff=0.2).align_to(grid_group, LEFT).shift(LEFT * 4.0)
            self.play(Transform(desc_lbl, desc_sin))

            hl_sin = get_highlight(13, color=BLUE) 
            self.play(ReplacementTransform(curr_hl, hl_sin))
            curr_hl = hl_sin

            val_c0 = math.sin(pos_idx * freq_vals[0])
            calc_sin = MathTex(
                rf"\sin(p_{pos_idx} \cdot \omega_0) = {val_c0:.2f}", 
                font_size=24, color=BLUE
            ).move_to(VIS_CENTER + UP * SCRATCH_PAD_TOP_Y)
            self.play(FadeIn(calc_sin))

            c0_rect = SurroundingRectangle(grid_squares[pos_idx][0], color=BLUE)
            c2_rect = SurroundingRectangle(grid_squares[pos_idx][2], color=BLUE)
            val_txt_c0 = MathTex(f"{val_c0:.2f}", font_size=18, color=BLUE).move_to(grid_squares[pos_idx][0])
            val_c2 = math.sin(pos_idx * freq_vals[2])
            val_txt_c2 = MathTex(f"{val_c2:.2f}", font_size=18, color=BLUE).move_to(grid_squares[pos_idx][2])

            self.play(Create(c0_rect), Create(c2_rect), Write(val_txt_c0), Write(val_txt_c2))
            self.wait(0.5)
            self.play(FadeOut(calc_sin), FadeOut(c0_rect), FadeOut(c2_rect))

            # --- COSINE ---
            # Update Description
            desc_cos = Tex(rf"Step 3: Fill Row $p_{{{pos_idx}}}$ at ODD cols ($d_1, d_3$) using Cosine.", font_size=22, color=RED)
            desc_cos.next_to(grid_title, DOWN, buff=0.2).align_to(grid_group, LEFT).shift(LEFT * 4.0)
            self.play(Transform(desc_lbl, desc_cos))

            hl_cos = get_highlight(16, color=RED)
            self.play(ReplacementTransform(curr_hl, hl_cos))
            curr_hl = hl_cos

            val_c1 = math.cos(pos_idx * freq_vals[1])
            calc_cos = MathTex(
                rf"\cos(p_{pos_idx} \cdot \omega_0) = {val_c1:.2f}", 
                font_size=24, color=RED
            ).move_to(VIS_CENTER + UP * SCRATCH_PAD_TOP_Y)
            self.play(FadeIn(calc_cos))

            c1_rect = SurroundingRectangle(grid_squares[pos_idx][1], color=RED)
            c3_rect = SurroundingRectangle(grid_squares[pos_idx][3], color=RED)
            val_txt_c1 = MathTex(f"{val_c1:.2f}", font_size=18, color=RED).move_to(grid_squares[pos_idx][1])
            val_c3 = math.cos(pos_idx * freq_vals[3])
            val_txt_c3 = MathTex(f"{val_c3:.2f}", font_size=18, color=RED).move_to(grid_squares[pos_idx][3])

            self.play(Create(c1_rect), Create(c3_rect), Write(val_txt_c1), Write(val_txt_c3))
            self.wait(0.5)
            self.play(FadeOut(calc_cos), FadeOut(c1_rect), FadeOut(c3_rect))

            self.play(FadeOut(row_rect))

        self.wait(1)
        if os.path.exists(code_filename):
            os.remove(code_filename)

%manim -qk -v warning PositionalEncodingVis

Manim Community v0.19.0